<a href="https://colab.research.google.com/github/Desire-in-tech/predictive-maintenance-ml/blob/task-09%2Ffailure-mode-enhancement/notebooks/task_09_predictive_maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 9 — Industrial Predictive Maintenance

Train a binary machine-failure classifier on the UCI AI4I 2020 dataset. The notebook downloads data directly from UCI, tunes the decision threshold on validation data to control false discoveries, evaluates once on a held-out test set, and saves the complete inference pipeline.

**Important:** AI4I provides a binary `Machine failure` target and separate failure-mode indicators; it is not a mutually exclusive five-class target. This notebook models the binary target.

In [1]:
# Install project dependencies
!pip -q install -r https://raw.githubusercontent.com/Desire-in-tech/predictive-maintenance-ml/task-09/failure-mode-enhancement/requirements.txt


In [2]:
# Clone the project branch on first run; update an existing checkout safely on reruns.
from pathlib import Path
import subprocess
import sys

repo_dir = Path('/content/predictive-maintenance-ml')
repo_url = 'https://github.com/Desire-in-tech/predictive-maintenance-ml.git'
branch = 'task-09/failure-mode-enhancement'

if (repo_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', branch], check=True)
else:
    subprocess.run(['git', 'clone', '--single-branch', '--branch', branch, repo_url, str(repo_dir)], check=True)

%cd /content/predictive-maintenance-ml
sys.path.insert(0, str(repo_dir.resolve()))


In [ ]:
# Download the dataset and inspect the target balance
from src.data_loader import load_ai4i, FEATURES, TARGET, FAILURE_MODES
import pandas as pd
df = load_ai4i()
display(df.head())
display(df[TARGET].value_counts().rename('count').to_frame())
print(f'Rows: {len(df):,}; failure prevalence: {df[TARGET].mean():.3%}')
print('Failure-mode positive counts:')
display(df[FAILURE_MODES].sum().sort_values(ascending=False).rename('positive_rows').to_frame())


In [ ]:
# Exploratory analysis: target distribution and sensor correlations
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(5, 3)); df[TARGET].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set(title='Machine failure class balance', xlabel='Machine failure (0=no, 1=yes)', ylabel='Rows'); plt.show()
plt.figure(figsize=(8, 6)); sns.heatmap(df.select_dtypes('number').corr(), cmap='coolwarm', center=0); plt.title('Numeric feature correlations'); plt.show()


In [5]:
# Train, select a validation threshold, evaluate on held-out test data, and export artifact
!python -m src.train --output-dir artifacts --model random_forest --min-recall 0.50


{
  "threshold": 0.51,
  "precision": 0.72,
  "recall": 0.5294117647058824,
  "f1": 0.6101694915254238,
  "false_discovery_rate": 0.28,
  "false_positive_rate": 0.007246376811594203,
  "roc_auc": 0.972491170381196,
  "average_precision": 0.7277334803392069,
  "confusion_matrix": {
    "tn": 1918,
    "fp": 14,
    "fn": 32,
    "tp": 36
  },
  "model": "random_forest",
  "dataset": "UCI AI4I 2020",
  "target": "Machine failure",
  "features": [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
  ],
  "validation_recall_floor": 0.5
}
Artifact: artifacts/predictive_maintenance_pipeline.joblib


In [6]:
# Review the test metrics and saved feature schema
import json
from pathlib import Path
metrics = json.loads(Path('artifacts/metrics.json').read_text())
display(pd.DataFrame([metrics]).T.rename(columns={0:'value'}))
print(Path('artifacts/feature_schema.json').read_text())


,value
threshold,0.51
precision,0.72
recall,0.529412
f1,0.610169
false_discovery_rate,0.28
false_positive_rate,0.007246
roc_auc,0.972491
average_precision,0.727733
confusion_matrix,"{'tn': 1918, 'fp': 14, 'fn': 32, 'tp': 36}"
model,random_forest


{
  "features": [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
  ],
  "target": "Machine failure",
  "threshold": 0.51
}


# Failure-mode classification and sensor analysis

The following cells train one classifier for each AI4I failure-mode indicator, select thresholds using validation data with FDR as the objective, evaluate on the held-out test split, and rank sensor correlations. These are independent multi-label predictions; more than one mode may be predicted for a row.

In [ ]:
# Train the five failure-mode models and save their evaluation reports
!python -m src.failure_modes --output-dir artifacts --min-recall 0.30

In [ ]:
# Rank sensor associations with overall failure and each failure mode
from src.analysis import save_sensor_failure_correlations
correlations = save_sensor_failure_correlations(df)
display(correlations.groupby('outcome', group_keys=False).head(5))
print('Saved ranked correlations to artifacts/sensor_failure_correlations.csv')

In [ ]:
# Optional: persist the model artifact and reports in your Google Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
destination = Path('/content/drive/MyDrive/predictive-maintenance-task09')
destination.mkdir(parents=True, exist_ok=True)
for filename in ['predictive_maintenance_pipeline.joblib', 'metrics.json', 'feature_schema.json', 'failure_modes_pipeline.joblib', 'failure_modes_metrics.json', 'failure_modes_schema.json', 'sensor_failure_correlations.csv']:
    shutil.copy2(Path('artifacts') / filename, destination / filename)
print('Saved versioned Task 9 outputs to:', destination)


In [8]:
# Smoke-test the same inference wrapper that Task 10 will reuse
from src.predict import PredictiveMaintenanceModel
model = PredictiveMaintenanceModel('artifacts/predictive_maintenance_pipeline.joblib')
sample = df[FEATURES].head(3).to_dict(orient='records')
model.predict(sample)


[{'failure_probability': 0.0, 'predicted_failure': False, 'threshold': 0.51},
 {'failure_probability': 0.0022515782090250175,
  'predicted_failure': False,
  'threshold': 0.51},
 {'failure_probability': 0.013541240609430193,
  'predicted_failure': False,
  'threshold': 0.51}]

In [ ]:
# Smoke-test the enhanced inference interface used by Task 10
enhanced_model = PredictiveMaintenanceModel(
    'artifacts/predictive_maintenance_pipeline.joblib',
    failure_modes_artifact_path='artifacts/failure_modes_pipeline.joblib'
)
enhanced_model.predict_with_failure_modes(sample)

In [ ]:
# Run pytest with PYTHONPATH set correctly to resolve import issues
!PYTHONPATH=. pytest -v